In [6]:
import requests
import zipfile
import io

url = "https://ndownloader.figshare.com/files/14464685"

response = requests.get(url, timeout=180)
response.raise_for_status()

archive = zipfile.ZipFile(io.BytesIO(response.content))

print("Files available:")
print(archive.namelist())

Files available:
['events_England.json', 'events_European_Championship.json', 'events_France.json', 'events_Germany.json', 'events_Italy.json', 'events_Spain.json', 'events_World_Cup.json']


In [7]:
import json
import pandas as pd

match_id = 2499943

# Read the England events from the ZIP already downloaded
filename = next(
    name for name in archive.namelist()
    if name.endswith("events_England.json")
)

with archive.open(filename) as file:
    england_events = json.load(file)

# Keep only events from Liverpool 4-3 Manchester City
match_events = [
    event for event in england_events
    if event["matchId"] == match_id
]

print("Match ID:", match_id)
print("Number of events:", len(match_events))

# Preview the event data
df = pd.DataFrame(match_events)
display(df.head())

Match ID: 2499943
Number of events: 1876


,eventId,subEventName,tags,playerId,positions,matchId,eventName,teamId,matchPeriod,eventSec,subEventId,id
0,8,Simple pass,[{'id': 1801}],15808,"[{'y': 49, 'x': 49}, {'y': 41, 'x': 31}]",2499943,Pass,1612,1H,2.528085,85,220933014
1,8,Simple pass,[{'id': 1801}],25393,"[{'y': 41, 'x': 31}, {'y': 70, 'x': 34}]",2499943,Pass,1612,1H,6.517082,85,220933015
2,8,Simple pass,[{'id': 1801}],14870,"[{'y': 70, 'x': 34}, {'y': 41, 'x': 28}]",2499943,Pass,1612,1H,10.732865,85,220933016
3,8,Simple pass,[{'id': 1801}],25393,"[{'y': 41, 'x': 28}, {'y': 50, 'x': 7}]",2499943,Pass,1612,1H,14.110714,85,220933017
4,8,Launch,[{'id': 1802}],15175,"[{'y': 50, 'x': 7}, {'y': 9, 'x': 61}]",2499943,Pass,1612,1H,16.587550,84,220933018


In [8]:
print("Total match events:", len(match_events))
print("Match IDs:", df["matchId"].unique())
print("Event types:")
print(df["eventName"].value_counts())

Total match events: 1876
Match IDs: [2499943]
Event types:
eventName
Pass                       984
Duel                       558
Others on the ball         126
Free Kick                   80
Interruption                58
Shot                        27
Foul                        18
Save attempt                11
Goalkeeper leaving line      8
Offside                      6
Name: count, dtype: int64


In [9]:
display(
    df[["playerId", "eventName", "eventSec", "positions"]].head(10)
)

,playerId,eventName,eventSec,positions
0,15808,Pass,2.528085,"[{'y': 49, 'x': 49}, {'y': 41, 'x': 31}]"
1,25393,Pass,6.517082,"[{'y': 41, 'x': 31}, {'y': 70, 'x': 34}]"
2,14870,Pass,10.732865,"[{'y': 70, 'x': 34}, {'y': 41, 'x': 28}]"
3,25393,Pass,14.110714,"[{'y': 41, 'x': 28}, {'y': 50, 'x': 7}]"
4,15175,Pass,16.587550,"[{'y': 50, 'x': 7}, {'y': 9, 'x': 61}]"
5,25747,Offside,19.464496,"[{'y': 9, 'x': 61}, {'y': 0, 'x': 0}]"
6,8277,Free Kick,35.997091,"[{'y': 89, 'x': 35}, {'y': 74, 'x': 18}]"
7,9380,Pass,38.497027,"[{'y': 74, 'x': 18}, {'y': 45, 'x': 12}]"
8,71654,Pass,41.991763,"[{'y': 45, 'x': 12}, {'y': 25, 'x': 14}]"
9,70086,Pass,43.942339,"[{'y': 25, 'x': 14}, {'y': 5, 'x': 12}]"


In [10]:
import requests
import json
import pandas as pd

players_url = "https://raw.githubusercontent.com/koenvo/wyscout-soccer-match-event-dataset/main/players.json"

response = requests.get(players_url, timeout=60)

print("HTTP status:", response.status_code)

if response.ok:
    players = response.json()
    players_df = pd.DataFrame(players)

    display(
        players_df[
            players_df["shortName"].str.contains(
                "Salah|Mané|Mane|Firmino",
                case=False,
                na=False
            )
        ][["wyId", "shortName"]]
    )
else:
    print("Player file unavailable. We'll use the original Wyscout player reference.")

HTTP status: 404
Player file unavailable. We'll use the original Wyscout player reference.


In [11]:
import requests
import pandas as pd

players_url = "https://ndownloader.figshare.com/files/15073721"

response = requests.get(players_url, timeout=60)
response.raise_for_status()

players = response.json()
players_df = pd.DataFrame(players)

print("Total players:", len(players_df))
print("Columns:", players_df.columns.tolist())


Total players: 3603
Columns: ['passportArea', 'weight', 'firstName', 'middleName', 'lastName', 'currentTeamId', 'birthDate', 'height', 'role', 'birthArea', 'wyId', 'foot', 'shortName', 'currentNationalTeamId']


In [13]:
# Find all players whose names contain Mane or Mané
display(
    players_df[
        players_df["shortName"].str.contains(
            "Man", case=False, na=False
        )
    ][["wyId", "shortName"]]
)

,wyId,shortName
2,393230,O. Mandanda
20,114,K. Strootman
86,492782,J. Klinsmann
109,263802,Y. Tielemans
116,231139,K. Coman
...,...,...
3378,229198,H. Manzala
3510,503185,J. Vagnoman
3519,83970,V. G\u0103man
3535,1735,H. Hermannsson


In [14]:
match_player_ids = set(df["playerId"].unique())

mane_candidates = players_df[
    players_df["shortName"].str.contains(
        "Man", case=False, na=False
    )
]

display(
    mane_candidates[
        mane_candidates["wyId"].isin(match_player_ids)
    ][["wyId", "shortName"]]
)

,wyId,shortName
1800,25747,S. Man\u00e9


In [15]:
# Match the exact player names
names = ["Mohamed Salah", "Roberto Firmino", "Sadio Mané"]

# Find players who participated in this match
match_ids = set(df["playerId"].unique())

confirmed_players = players_df[
    players_df["wyId"].isin(match_ids)
    & players_df["shortName"].str.contains(
        r"Salah|Firmino|S\. Man",
        case=False,
        na=False
    )
].copy()

display(confirmed_players[["wyId", "shortName"]])

,wyId,shortName
1191,15808,Roberto Firmino
1719,120353,Mohamed Salah
1800,25747,S. Man\u00e9


In [16]:
# Give the three players clean, readable names
name_map = {
    "Salah": "Mohamed Salah",
    "Firmino": "Roberto Firmino",
    "Man": "Sadio Mané"
}

confirmed_players = confirmed_players.copy()

def clean_player_name(name):
    if "Salah" in name:
        return "Mohamed Salah"
    elif "Firmino" in name:
        return "Roberto Firmino"
    elif "Man" in name:
        return "Sadio Mané"
    return name

confirmed_players["Player"] = confirmed_players["shortName"].apply(
    clean_player_name
)

display(confirmed_players[["wyId", "Player"]])

,wyId,Player
1191,15808,Roberto Firmino
1719,120353,Mohamed Salah
1800,25747,Sadio Mané


In [17]:
player_events = df[
    df["playerId"].isin(confirmed_players["wyId"])
].copy()

player_events["Player"] = player_events["playerId"].map(
    confirmed_players.set_index("wyId")["Player"]
)

print(player_events["Player"].value_counts())

Player
Roberto Firmino    77
Sadio Mané         68
Mohamed Salah      48
Name: count, dtype: int64


In [18]:
print("Event columns:")
print(df.columns.tolist())

print("\nEvent types:")
print(df["eventName"].value_counts())

print("\nMatch periods:")
print(df["matchPeriod"].value_counts())

print("\nSample event:")
import json
print(json.dumps(match_events[0], indent=2))

Event columns:
['eventId', 'subEventName', 'tags', 'playerId', 'positions', 'matchId', 'eventName', 'teamId', 'matchPeriod', 'eventSec', 'subEventId', 'id']

Event types:
eventName
Pass                       984
Duel                       558
Others on the ball         126
Free Kick                   80
Interruption                58
Shot                        27
Foul                        18
Save attempt                11
Goalkeeper leaving line      8
Offside                      6
Name: count, dtype: int64

Match periods:
matchPeriod
1H    939
2H    937
Name: count, dtype: int64

Sample event:
{
  "eventId": 8,
  "subEventName": "Simple pass",
  "tags": [
    {
      "id": 1801
    }
  ],
  "playerId": 15808,
  "positions": [
    {
      "y": 49,
      "x": 49
    },
    {
      "y": 41,
      "x": 31
    }
  ],
  "matchId": 2499943,
  "eventName": "Pass",
  "teamId": 1612,
  "matchPeriod": "1H",
  "eventSec": 2.528084999999976,
  "subEventId": 85,
  "id": 220933014
}


In [19]:
# Start with all 1,876 events
events_clean = df.copy()

def get_coordinate(positions, index, axis):
    if (
        isinstance(positions, list)
        and len(positions) > index
        and isinstance(positions[index], dict)
    ):
        return positions[index].get(axis)
    return None

# Starting coordinates
events_clean["start_x"] = events_clean["positions"].apply(
    lambda p: get_coordinate(p, 0, "x")
)

events_clean["start_y"] = events_clean["positions"].apply(
    lambda p: get_coordinate(p, 0, "y")
)

# Ending coordinates
events_clean["end_x"] = events_clean["positions"].apply(
    lambda p: get_coordinate(p, 1, "x")
)

events_clean["end_y"] = events_clean["positions"].apply(
    lambda p: get_coordinate(p, 1, "y")
)

display(
    events_clean[
        ["eventName", "playerId", "start_x", "start_y", "end_x", "end_y"]
    ].head(10)
)

,eventName,playerId,start_x,start_y,end_x,end_y
0,Pass,15808,49,49,31,41
1,Pass,25393,31,41,34,70
2,Pass,14870,34,70,28,41
3,Pass,25393,28,41,7,50
4,Pass,15175,7,50,61,9
5,Offside,25747,61,9,0,0
6,Free Kick,8277,35,89,18,74
7,Pass,9380,18,74,12,45
8,Pass,71654,12,45,14,25
9,Pass,70086,14,25,12,5


In [20]:
# Preserve the original time within each half
events_clean["period_seconds"] = events_clean["eventSec"]

# Calculate continuous match time
events_clean["match_seconds"] = (
    events_clean["period_seconds"]
    + events_clean["matchPeriod"].map({
        "1H": 0,
        "2H": 45 * 60
    })
)

# Convert to minutes
events_clean["match_minute"] = (
    events_clean["match_seconds"] / 60
).round(2)

# Create a readable minute label
events_clean["minute_label"] = (
    events_clean["match_minute"].astype(int).astype(str) + "'"
)

# Preview the results
display(
    events_clean[
        [
            "eventName",
            "matchPeriod",
            "eventSec",
            "match_seconds",
            "match_minute",
            "minute_label"
        ]
    ].iloc[[0, 100, 938, 939, 1000, 1875]]
)

,eventName,matchPeriod,eventSec,match_seconds,match_minute,minute_label
0,Pass,1H,2.528085,2.528085,0.04,0'
100,Pass,1H,243.979952,243.979952,4.07,4'
938,Pass,1H,2827.987627,2827.987627,47.13,47'
939,Pass,2H,3.853006,2703.853006,45.06,45'
1000,Pass,2H,127.762587,2827.762587,47.13,47'
1875,Pass,2H,2993.679122,5693.679122,94.89,94'


In [21]:
# Extract tag IDs into a list
def extract_tag_ids(tags):
    if not isinstance(tags, list):
        return []
    return [
        tag["id"]
        for tag in tags
        if isinstance(tag, dict) and "id" in tag
    ]

events_clean["tag_ids"] = events_clean["tags"].apply(
    extract_tag_ids
)

# Create event flags
events_clean["is_accurate"] = events_clean["tag_ids"].apply(
    lambda tags: 1801 in tags
)

events_clean["is_inaccurate"] = events_clean["tag_ids"].apply(
    lambda tags: 1802 in tags
)

events_clean["is_goal"] = events_clean["tag_ids"].apply(
    lambda tags: 101 in tags
)

events_clean["is_assist"] = events_clean["tag_ids"].apply(
    lambda tags: 301 in tags
)

# Check the results
print("Accurate-tagged events:", events_clean["is_accurate"].sum())
print("Inaccurate-tagged events:", events_clean["is_inaccurate"].sum())
print("Goal-tagged events:", events_clean["is_goal"].sum())
print("Assist-tagged events:", events_clean["is_assist"].sum())

display(
    events_clean[
        [
            "eventName",
            "subEventName",
            "tag_ids",
            "is_accurate",
            "is_goal",
            "is_assist"
        ]
    ].head(10)
)

Accurate-tagged events: 1278
Inaccurate-tagged events: 410
Goal-tagged events: 14
Assist-tagged events: 2


,eventName,subEventName,tag_ids,is_accurate,is_goal,is_assist
0,Pass,Simple pass,[1801],True,False,False
1,Pass,Simple pass,[1801],True,False,False
2,Pass,Simple pass,[1801],True,False,False
3,Pass,Simple pass,[1801],True,False,False
4,Pass,Launch,[1802],False,False,False
5,Offside,,[],False,False,False
6,Free Kick,Free Kick,[1801],True,False,False
7,Pass,Simple pass,[1801],True,False,False
8,Pass,Simple pass,[1801],True,False,False
9,Pass,Simple pass,[1801],True,False,False


In [22]:
print("Accurate:", events_clean["is_accurate"].sum())
print("Inaccurate:", events_clean["is_inaccurate"].sum())
print("Goals:", events_clean["is_goal"].sum())
print("Assists:", events_clean["is_assist"].sum())

# Show every goal-tagged event
display(
    events_clean.loc[
        events_clean["is_goal"],
        [
            "playerId",
            "teamId",
            "eventName",
            "subEventName",
            "matchPeriod",
            "match_minute",
            "tag_ids"
        ]
    ]
)

Accurate: 1278
Inaccurate: 410
Goals: 14
Assists: 2


,playerId,teamId,eventName,subEventName,matchPeriod,match_minute,tag_ids
211,7868,1612,Shot,Shot,1H,8.81,"[101, 402, 201, 1205, 1801]"
212,71654,1625,Save attempt,Save attempt,1H,8.83,"[101, 1205, 1802]"
802,245364,1625,Shot,Shot,1H,40.06,"[101, 401, 201, 1204, 1801]"
803,15175,1612,Save attempt,Reflexes,1H,40.09,"[101, 1204, 1802]"
1248,15808,1612,Shot,Shot,2H,58.95,"[101, 402, 201, 1206, 1801]"
1249,71654,1625,Save attempt,Reflexes,2H,58.98,"[101, 1206, 1802]"
1274,25747,1612,Shot,Shot,2H,61.07,"[101, 401, 201, 1208, 1801]"
1275,71654,1625,Save attempt,Reflexes,2H,61.10,"[101, 1208, 1802]"
1374,120353,1612,Shot,Shot,2H,67.06,"[101, 401, 1401, 201, 1201, 1801]"
1375,71654,1625,Save attempt,Save attempt,2H,67.11,"[101, 1201, 1802]"


In [24]:
goal_events = events_clean[
    events_clean["is_goal"]
].copy()

print("Goal-tagged events by type:")
print(goal_events["eventName"].value_counts())

# Sort first, then select columns for display
goal_events = goal_events.sort_values(
    ["matchPeriod", "eventSec"]
)

display(
    goal_events[
        [
            "id",
            "playerId",
            "teamId",
            "eventName",
            "subEventName",
            "matchPeriod",
            "match_minute",
            "tag_ids"
        ]
    ]
)

Goal-tagged events by type:
eventName
Shot            7
Save attempt    7
Name: count, dtype: int64


,id,playerId,teamId,eventName,subEventName,matchPeriod,match_minute,tag_ids
211,220933208,7868,1612,Shot,Shot,1H,8.81,"[101, 402, 201, 1205, 1801]"
212,220933268,71654,1625,Save attempt,Save attempt,1H,8.83,"[101, 1205, 1802]"
802,220933931,245364,1625,Shot,Shot,1H,40.06,"[101, 401, 201, 1204, 1801]"
803,220933739,15175,1612,Save attempt,Reflexes,1H,40.09,"[101, 1204, 1802]"
1248,220934280,15808,1612,Shot,Shot,2H,58.95,"[101, 402, 201, 1206, 1801]"
1249,220934316,71654,1625,Save attempt,Reflexes,2H,58.98,"[101, 1206, 1802]"
1274,220934313,25747,1612,Shot,Shot,2H,61.07,"[101, 401, 201, 1208, 1801]"
1275,220934340,71654,1625,Save attempt,Reflexes,2H,61.10,"[101, 1208, 1802]"
1374,220934410,120353,1612,Shot,Shot,2H,67.06,"[101, 401, 1401, 201, 1201, 1801]"
1375,220934459,71654,1625,Save attempt,Save attempt,2H,67.11,"[101, 1201, 1802]"


In [25]:
events_clean["is_goal_scored"] = (
    events_clean["is_goal"]
    & events_clean["eventName"].eq("Shot")
)

events_clean["is_goal_conceded_event"] = (
    events_clean["is_goal"]
    & events_clean["eventName"].eq("Save attempt")
)

print("Goals scored:", events_clean["is_goal_scored"].sum())
print(
    "Goal-associated goalkeeper events:",
    events_clean["is_goal_conceded_event"].sum()
)

print("\nGoals by team ID:")
print(
    events_clean.loc[events_clean["is_goal_scored"]]
    .groupby("teamId")
    .size()
)

Goals scored: 7
Goal-associated goalkeeper events: 7

Goals by team ID:
teamId
1612    4
1625    3
dtype: int64


In [26]:
# Map team IDs to names
team_map = {
    1612: "Liverpool",
    1625: "Manchester City"
}

events_clean["team_name"] = events_clean["teamId"].map(team_map)

# Build player reference table
player_lookup = (
    players_df[["wyId", "shortName"]]
    .drop_duplicates(subset=["wyId"])
    .rename(columns={
        "wyId": "playerId",
        "shortName": "player_name"
    })
)

# Join player names onto all match events
events_clean = events_clean.merge(
    player_lookup,
    on="playerId",
    how="left",
    validate="many_to_one"
)

# Clean literal Unicode escape sequences
def clean_name(name):
    if not isinstance(name, str):
        return name
    return name.replace("\\u00e9", "é")

events_clean["player_name"] = events_clean["player_name"].apply(clean_name)

# Preview
display(
    events_clean[
        [
            "team_name",
            "player_name",
            "eventName",
            "subEventName",
            "match_minute",
            "start_x",
            "start_y"
        ]
    ].head(10)
)

print("Total events:", len(events_clean))
print("Events missing player names:", events_clean["player_name"].isna().sum())

,team_name,player_name,eventName,subEventName,match_minute,start_x,start_y
0,Liverpool,Roberto Firmino,Pass,Simple pass,0.04,49,49
1,Liverpool,D. Lovren,Pass,Simple pass,0.11,31,41
2,Liverpool,J. Matip,Pass,Simple pass,0.18,34,70
3,Liverpool,D. Lovren,Pass,Simple pass,0.24,28,41
4,Liverpool,L. Karius,Pass,Launch,0.28,7,50
5,Liverpool,S. Mané,Offside,,0.32,61,9
6,Manchester City,K. Walker,Free Kick,Free Kick,0.60,35,89
7,Manchester City,J. Stones,Pass,Simple pass,0.64,18,74
8,Manchester City,Ederson,Pass,Simple pass,0.70,12,45
9,Manchester City,N. Otamendi,Pass,Simple pass,0.73,14,25


Total events: 1876
Events missing player names: 143


In [27]:
missing_players = events_clean[
    events_clean["player_name"].isna()
].copy()

print("Missing player names by event type:")
print(missing_players["eventName"].value_counts())

print("\nMissing player IDs:")
print(missing_players["playerId"].value_counts().head(15))

print("\nMissing player names by team:")
print(missing_players["team_name"].value_counts(dropna=False))

print("\nMissing player IDs that are not zero:")
print(
    missing_players.loc[
        missing_players["playerId"].ne(0),
        "playerId"
    ].unique()
)

Missing player names by event type:
eventName
Duel            80
Interruption    58
Free Kick        5
Name: count, dtype: int64

Missing player IDs:
playerId
0    143
Name: count, dtype: int64

Missing player names by team:
team_name
Manchester City    104
Liverpool           39
Name: count, dtype: int64

Missing player IDs that are not zero:
[]


In [28]:
# 1. Row counts and duplicate IDs
print("Total events:", len(events_clean))
print("Unique event IDs:", events_clean["id"].nunique())
print("Duplicate event IDs:", events_clean["id"].duplicated().sum())

# 2. Missing coordinates
print("\nMissing coordinates:")
print(
    events_clean[
        ["start_x", "start_y", "end_x", "end_y"]
    ].isna().sum()
)

# 3. Coordinate ranges
print("\nCoordinate ranges:")
print(
    events_clean[
        ["start_x", "start_y", "end_x", "end_y"]
    ].agg(["min", "max"])
)

# 4. Check for invalid coordinates
coordinate_cols = ["start_x", "start_y", "end_x", "end_y"]

invalid_coords = (
    (events_clean[coordinate_cols] < 0)
    | (events_clean[coordinate_cols] > 100)
).any(axis=1)

print("\nEvents with invalid coordinates:", invalid_coords.sum())

# 5. Match clock ranges
print("\nMatch time by half:")
print(
    events_clean.groupby("matchPeriod")["match_minute"]
    .agg(["min", "max"])
)

Total events: 1876
Unique event IDs: 1876
Duplicate event IDs: 0

Missing coordinates:
start_x    0
start_y    0
end_x      0
end_y      0
dtype: int64

Coordinate ranges:
     start_x  start_y  end_x  end_y
min        0        0      0      0
max      100      100    100    100

Events with invalid coordinates: 0

Match time by half:
               min    max
matchPeriod              
1H            0.04  47.13
2H           45.06  94.89


In [29]:
# Add match metadata
events_clean["competition"] = "Premier League"
events_clean["season"] = "2017/18"
events_clean["match_date"] = "2018-01-14"
events_clean["match_name"] = "Liverpool 4-3 Manchester City"

# Preserve unassigned player IDs
events_clean["player_assigned"] = events_clean["playerId"].ne(0)

# Convert tag lists to JSON strings for portable export
import json

events_clean["tags_json"] = events_clean["tags"].apply(
    lambda value: json.dumps(value, ensure_ascii=False)
)

# Select our standardized event columns
master_columns = [
    "id", "matchId", "competition", "season",
    "match_date", "match_name",
    "teamId", "team_name",
    "playerId", "player_name", "player_assigned",
    "eventId", "subEventId",
    "eventName", "subEventName",
    "matchPeriod", "eventSec",
    "match_seconds", "match_minute", "minute_label",
    "start_x", "start_y", "end_x", "end_y",
    "is_accurate", "is_inaccurate",
    "is_goal", "is_goal_scored",
    "is_goal_conceded_event", "is_assist",
    "tags_json"
]

master_events = events_clean[master_columns].copy()

print("Master rows:", len(master_events))
print("Master columns:", len(master_events.columns))

display(master_events.head())

Master rows: 1876
Master columns: 31


,id,matchId,competition,season,match_date,match_name,teamId,team_name,playerId,player_name,...,start_y,end_x,end_y,is_accurate,is_inaccurate,is_goal,is_goal_scored,is_goal_conceded_event,is_assist,tags_json
0,220933014,2499943,Premier League,2017/18,2018-01-14,Liverpool 4-3 Manchester City,1612,Liverpool,15808,Roberto Firmino,...,49,31,41,True,False,False,False,False,False,"[{""id"": 1801}]"
1,220933015,2499943,Premier League,2017/18,2018-01-14,Liverpool 4-3 Manchester City,1612,Liverpool,25393,D. Lovren,...,41,34,70,True,False,False,False,False,False,"[{""id"": 1801}]"
2,220933016,2499943,Premier League,2017/18,2018-01-14,Liverpool 4-3 Manchester City,1612,Liverpool,14870,J. Matip,...,70,28,41,True,False,False,False,False,False,"[{""id"": 1801}]"
3,220933017,2499943,Premier League,2017/18,2018-01-14,Liverpool 4-3 Manchester City,1612,Liverpool,25393,D. Lovren,...,41,7,50,True,False,False,False,False,False,"[{""id"": 1801}]"
4,220933018,2499943,Premier League,2017/18,2018-01-14,Liverpool 4-3 Manchester City,1612,Liverpool,15175,L. Karius,...,50,61,9,False,True,False,False,False,False,"[{""id"": 1802}]"


In [30]:
from pathlib import Path

output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "liverpool_mancity_2018_events.csv"

master_events.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("Saved CSV:", csv_path.resolve())

Saved CSV: /Users/ray97/Documents/Footballeventanalytics/data/processed/liverpool_mancity_2018_events.csv


In [32]:
%pip install nbformat

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 78 kB 2.0 MB/s eta 0:00:011
     |████████████████████████████████| 90 kB 46.1 MB/s eta 0:00:01
     |████████████████████████████████| 67 kB 13.6 MB/s eta 0:00:01
     |████████████████████████████████| 354 kB 10.6 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [33]:
import nbformat

notebook_path = "football_events.ipynb"

try:
    nb = nbformat.read(notebook_path, as_version=4)
    nbformat.validate(nb)
    print("Notebook is valid!")
except Exception as e:
    print("Validation error:", e)

Notebook is valid!
